# Módulo 07 · Texto y datos sucios

**Curso de pandas intermedio → avanzado** · Dataset: *Olist* (muestras en `data/`)

## Objetivos
1. Usar el accessor **`.str`** sobre el dtype `str` de pandas 3, incluidos los faltantes.
2. Limpiar texto: espacios, mayúsculas, **acentos**, saltos de línea y puntuación.
3. Escribir **expresiones regulares** con `contains`, `fullmatch`, `extract`, `extractall`, `replace` y `count`.
4. Construir **llaves canónicas** para que "São Paulo", "sao paulo " y "SAO-PAULO" sean lo mismo.
5. Convertir **números escritos como texto** (formato brasileño `R$ 1.234,56`).
6. Escribir **reglas de validación** y un reporte de calidad reutilizable.

| Sección | Tema |
|---|---|
| 0 | Setup y carga |
| 1 | El accessor `.str` en pandas 3 |
| 2 | Limpieza básica |
| 3 | Expresiones regulares |
| 4 | `split`, `partition` y `cat` |
| 5 | Llaves canónicas y coincidencias aproximadas |
| 6 | Valores "casi faltantes" y números sucios |
| 7 | Validación y reporte de calidad |
| 8 | Rendimiento con texto |
| ✍️ | Ejercicios · ✅ Soluciones |

## 0 · Setup y carga

In [ ]:
import re
import unicodedata
from difflib import get_close_matches
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)


def pendiente(*valores) -> bool:
  # True si algún valor sigue siendo `...` (ejercicio sin resolver)
  return any(v is ... for v in valores)


print("pandas", pd.__version__)

In [ ]:
reviews = pd.read_csv(
  DATA / "olist_order_reviews_dataset.csv",
  usecols=["review_id", "order_id", "review_score", "review_comment_title", "review_comment_message"],
)
customers = pd.read_csv(DATA / "olist_customers_dataset.csv",
                        dtype={"customer_zip_code_prefix": "str"})
sellers = pd.read_csv(DATA / "olist_sellers_dataset.csv",
                      dtype={"seller_zip_code_prefix": "str"})
geo = pd.read_csv(DATA / "olist_geolocation_dataset.csv",
                  dtype={"geolocation_zip_code_prefix": "str"})
traduccion = pd.read_csv(DATA / "product_category_name_translation.csv")

for df, col in [(customers, "customer_zip_code_prefix"), (sellers, "seller_zip_code_prefix"),
                (geo, "geolocation_zip_code_prefix")]:
  df[col] = df[col].str.zfill(5)

## 1 · El accessor `.str` en pandas 3

Todo método de `.str` es **vectorizado** y **respeta los faltantes**: si el valor es `NaN`, el resultado también lo es (no hay que filtrar antes).

In [ ]:
titulos = reviews["review_comment_title"]
print("dtype:", titulos.dtype, "| faltantes:", titulos.isna().sum(), "de", len(titulos))
titulos.str.upper().head(4)

Ojo con los métodos que devuelven **booleanos**: con faltantes, el resultado puede no ser `bool` puro. Al filtrar, usa `na=False`:

In [ ]:
tiene_recom = titulos.str.contains("recom", case=False, na=False)
print(tiene_recom.dtype, "|", tiene_recom.sum(), "títulos mencionan 'recom'")

Métodos que conviene tener a mano:

| Grupo | Métodos |
|---|---|
| Forma | `lower`, `upper`, `title`, `casefold`, `strip`, `lstrip`, `rstrip`, `pad`, `zfill` |
| Búsqueda | `contains`, `startswith`, `endswith`, `match`, `fullmatch`, `find`, `count` |
| Extracción | `extract`, `extractall`, `findall`, `slice`, `get`, `[i:j]` |
| Transformación | `replace`, `removeprefix`, `removesuffix`, `normalize`, `translate` |
| Estructura | `split`, `rsplit`, `partition`, `join`, `cat`, `len`, `get_dummies` |

## 2 · Limpieza básica

Los títulos de las reseñas son un buen laboratorio: la misma idea escrita de muchas formas.

In [ ]:
titulos.value_counts().head(15)

Construimos una limpieza **por pasos**, cada uno con un propósito claro:

In [ ]:
def quitar_acentos(s: pd.Series) -> pd.Series:
  # NFKD separa la letra de su acento ("ó" -> "o" + "´"); luego descartamos lo no-ASCII
  return s.str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("ascii")


paso = pd.DataFrame({"original": titulos.dropna()})
paso["strip_lower"] = paso["original"].str.strip().str.lower()
paso["sin_acentos"] = quitar_acentos(paso["strip_lower"])
paso["sin_puntuacion"] = paso["sin_acentos"].str.replace(r"[^\w\s]", "", regex=True).str.strip()
paso["espacios"] = paso["sin_puntuacion"].str.replace(r"\s+", " ", regex=True)

paso.nunique()

In [ ]:
paso[paso["original"].str.contains("timo|bom", case=False)].drop_duplicates("original").head(8)

> 💡 `str.normalize` + `encode("ascii", errors="ignore")` elimina acentos, pero también **cualquier** carácter no latino (emojis, ñ → n, ç → c). Para portugués y español es lo que queremos; para otros idiomas, cuidado.

### Saltos de línea y espacios raros

Los mensajes de reseña traen `\r\n` y espacios al final. `\s` en regex cubre espacios, tabs y saltos de línea:

In [ ]:
msg = reviews["review_comment_message"]
print("Con \\r o \\n          :", msg.str.contains(r"[\r\n]", na=False).sum())
print("Con espacios extremos :", (msg != msg.str.strip()).sum())

msg_limpio = msg.str.replace(r"\s+", " ", regex=True).str.strip()
msg[msg.str.contains("\r", na=False)].iloc[0], msg_limpio[msg.str.contains("\r", na=False)].iloc[0]

## 3 · Expresiones regulares

Repaso rápido de regex (usa siempre *raw strings* `r"..."`):

| Patrón | Significa |
|---|---|
| `\d`, `\w`, `\s` | dígito, carácter de palabra, espacio |
| `[abc]`, `[^abc]` | uno de / ninguno de |
| `+`, `*`, `?`, `{n,m}` | repeticiones |
| `^`, `$`, `\b` | inicio, fin, frontera de palabra |
| `(…)`, `(?P<nombre>…)` | grupo, grupo con nombre |
| `a|b` | alternativa |
| `(?i)` | sin distinguir mayúsculas |

### `contains`, `match`, `fullmatch`

- `contains`: el patrón aparece **en cualquier parte**.
- `match`: el patrón aparece **al inicio**.
- `fullmatch`: el patrón describe **todo** el texto. Es el correcto para **validar formatos**.

In [ ]:
prueba = pd.Series(["12345", "1234", "12345-678", "ab123"])
pd.DataFrame({
  "valor": prueba,
  "contains": prueba.str.contains(r"\d{5}"),
  "match": prueba.str.match(r"\d{5}"),
  "fullmatch": prueba.str.fullmatch(r"\d{5}"),
})

### `extract`: sacar partes con grupos

Cada grupo se vuelve una columna; con grupos con nombre, las columnas se llaman así. Si no hay coincidencia, `NaN`.

In [ ]:
fechas_en_texto = msg.str.extract(r"(?P<dia>\d{1,2})/(?P<mes>\d{1,2})/(?P<anio>\d{4})")
fechas_en_texto.dropna()

In [ ]:
# Números mencionados en los mensajes, con su contexto inmediato
msg.str.extract(r"(?P<numero>\d+)\s*(?P<palabra>[a-zA-Zçãõé]+)?").dropna(subset=["numero"]).head(8)

### `extractall` y `findall`: todas las coincidencias

`extract` toma sólo la **primera**. `extractall` devuelve **todas**, con un `MultiIndex` (fila original, número de coincidencia). `findall` devuelve listas (combínalo con `explode`, Módulo 04).

In [ ]:
signos = msg.str.extractall(r"(?P<signo>[!?]{2,})")
signos.head()

In [ ]:
signos.groupby(level=0).size().describe()          # cuántas "exclamaciones múltiples" por mensaje

### `replace` con grupos y funciones

In [ ]:
# Reordenar fechas dd/mm/aaaa -> aaaa-mm-dd usando referencias a grupos
msg.str.replace(r"(\d{1,2})/(\d{1,2})/(\d{4})", r"\3-\2-\1", regex=True)[fechas_en_texto["dia"].notna()]

In [ ]:
# replace acepta una función que recibe el objeto Match
censura = msg.str.replace(r"\b\d+\b", lambda m: "#" * len(m.group()), regex=True)
censura[msg.str.contains(r"\d", na=False)].head(3)

### `count` y clasificación por temas

Una forma barata (y muy usada) de etiquetar texto: una regex por tema. Normalizamos **antes** para no tener que escribir variantes con y sin acento.

In [ ]:
msg_norm = quitar_acentos(msg.str.lower())

TEMAS = {
  "entrega_ok": r"\b(?:no prazo|antes do prazo|chegou rapido|entrega rapida|super rapid)",
  "no_recibido": r"\b(?:nao recebi|nao chegou|nao foi entregue|ainda nao)",
  "defecto": r"\b(?:quebrad|defeito|nao funcion|avaria|falsificad)",
  "recomienda": r"\brecomend",
}
etiquetas = pd.DataFrame({tema: msg_norm.str.contains(patron, na=False) for tema, patron in TEMAS.items()})
etiquetas.sum()

> 💡 Usamos grupos **no capturantes** `(?:...)`: con `contains` sólo nos interesa si hay coincidencia, y pandas avisa con un warning si el patrón tiene grupos capturantes que no se usarán.

In [ ]:
etiquetas.assign(score=reviews["review_score"]).groupby("score").mean().mul(100).round(0)

## 4 · `split`, `partition` y `cat`

In [ ]:
# expand=True devuelve columnas; n limita el número de cortes
traduccion["product_category_name"].str.split("_", n=1, expand=True).head()

In [ ]:
# .str.get / .str[i] sobre listas
partes = traduccion["product_category_name"].str.split("_")
pd.DataFrame({"primera": partes.str[0], "ultima": partes.str[-1], "n_partes": partes.str.len()}).head()

In [ ]:
# partition: siempre 3 columnas (antes, separador, después), sólo el primer corte
sellers["seller_city"].str.partition(" / ").loc[lambda d: d[1] != ""]

In [ ]:
# cat: concatenar columnas de texto (con separador y manejo de faltantes)
sellers["seller_city"].str.cat(sellers["seller_state"], sep=" - ").head(3)

## 5 · Llaves canónicas y coincidencias aproximadas

`customers` y `sellers` escriben las ciudades **sin acentos**; `geolocation`, **con** acentos. Comparemos para el mismo código postal:

In [ ]:
ciudad_geo = geo.groupby("geolocation_zip_code_prefix")["geolocation_city"].agg(lambda s: s.mode().iloc[0])

comp = customers.assign(ciudad_geo=customers["customer_zip_code_prefix"].map(ciudad_geo)).dropna(subset=["ciudad_geo"])
print("Coinciden tal cual:", (comp["customer_city"] == comp["ciudad_geo"]).mean().round(3))
comp.loc[comp["customer_city"] != comp["ciudad_geo"], ["customer_city", "ciudad_geo"]].drop_duplicates().head(8)

La solución: una función que lleve cualquier variante a una **llave canónica**. Se aplica a **ambos lados** antes de comparar o unir.

In [ ]:
def llave_canonica(s: pd.Series) -> pd.Series:
  return (
    quitar_acentos(s.str.lower())
    .str.replace(r"[-'’.]", " ", regex=True)     # guiones y apóstrofos como espacio
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
  )


pd.Series(["São Paulo", " sao  paulo", "SAO-PAULO", "santa bárbara d'oeste", "mogi-guaçu"]).pipe(llave_canonica)

In [ ]:
coinciden = llave_canonica(comp["customer_city"]) == llave_canonica(comp["ciudad_geo"])
print("Coinciden tras canonicalizar:", coinciden.mean().round(3))
comp.loc[~coinciden, ["customer_zip_code_prefix", "customer_city", "ciudad_geo"]]

En esta muestra **todas** las diferencias eran de acentos. En el dataset completo quedan además diferencias **reales** (un barrio contra su municipio, nombres alternativos) que ningún regex arregla: requieren una decisión o una tabla de correspondencias.

### Errores de escritura: coincidencia aproximada

Para errores de tipeo, la librería estándar trae `difflib.get_close_matches`, que compara contra una lista de valores de referencia. Probemos con las ciudades de vendedores que **no** aparecen en `geo` ni en `customers`:

In [ ]:
validas = sorted(set(llave_canonica(geo["geolocation_city"])) | set(llave_canonica(customers["customer_city"])))

sospechosas = sellers.loc[~llave_canonica(sellers["seller_city"]).isin(validas), "seller_city"].unique()
print(len(sospechosas), "ciudades de vendedores sin referencia")
pd.DataFrame({
  "escrita": sospechosas,
  "sugerencia": [next(iter(get_close_matches(c, validas, n=1, cutoff=0.85)), None) for c in sospechosas],
}).dropna()

¡Ambas sugerencias son **incorrectas**! Ribeirão Preto y Guariba son ciudades reales que simplemente no están en la lista de referencia de la muestra, así que `difflib` propone lo más parecido que encuentra (Ribeirão **Pires**, **Guaíba**).

> ⚠️ Usa la coincidencia aproximada para **sugerir** correcciones y revisarlas, nunca para aplicarlas a ciegas. Su calidad depende por completo de que la referencia sea completa.

Otra técnica útil cuando no tienes referencia: buscar **casi-duplicados dentro de la misma columna**. Un error de tipeo suele aparecer junto a la versión correcta:

In [ ]:
from difflib import SequenceMatcher
from itertools import combinations

unicas = sorted(llave_canonica(sellers["seller_city"]).unique())
pares = [(a, b, round(SequenceMatcher(None, a, b).ratio(), 3))
         for a, b in combinations(unicas, 2)
         if SequenceMatcher(None, a, b).ratio() >= 0.9]
pd.DataFrame(pares, columns=["ciudad_a", "ciudad_b", "similitud"])

Ahí aparece `riberao preto` junto a `ribeirao preto`: un error de tipeo que sí conviene corregir. (Comparar todos los pares crece como n²; para miles de valores únicos se usan librerías especializadas como `rapidfuzz`.)

## 6 · Valores "casi faltantes" y números sucios

En datos reales los faltantes llegan disfrazados: `""`, `" "`, `"N/A"`, `"null"`, `"-"`. Y los números llegan con formato local. Este ejemplo es **sintético** (escrito para el curso), pero imita lo que llega en exportaciones de sistemas brasileños:

In [ ]:
sucio = pd.DataFrame({
  "producto": ["A", "B", "C", "D", "E", "F", "G", "H"],
  "precio_txt": ["R$ 1.234,56", "89,90", " R$12,5 ", "N/A", "", "grátis", "1.000", None],
  "unidades_txt": ["3", " 2 ", "-", "10", "null", "1", "1.0", "x2"],
})
sucio

In [ ]:
CASI_NULOS = r"^\s*(|-|n/?a|null|none|nan)\s*$"

limpio = sucio.assign(
  precio_txt=sucio["precio_txt"].str.replace(CASI_NULOS, "", regex=True, case=False).replace("", np.nan),
  unidades_txt=sucio["unidades_txt"].str.replace(CASI_NULOS, "", regex=True, case=False).replace("", np.nan),
)
limpio

Para el formato brasileño (`.` miles, `,` decimales): quitar símbolo y espacios → quitar puntos de miles → coma por punto → `pd.to_numeric(errors="coerce")`, que convierte en `NaN` lo que siga sin ser número.

In [ ]:
def a_numero_br(s: pd.Series) -> pd.Series:
  t = (s.str.replace(r"R\$|\s", "", regex=True)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False))
  return pd.to_numeric(t, errors="coerce")


limpio.assign(precio=a_numero_br(limpio["precio_txt"]),
              unidades=pd.to_numeric(limpio["unidades_txt"].str.strip(), errors="coerce").astype("Int64"))

Observa los **casos frontera**: `"grátis"` debería ser 0 (regla de negocio, no de formato) y `"1.0"` en `unidades` se volvió `10` si usáramos `a_numero_br` (¡ambiguo!). Por eso conviene aplicar cada regla sólo a la columna que le corresponde y revisar lo que `errors="coerce"` convirtió en `NaN`.

También puedes declarar los casi-nulos y el formato **al leer**:

```python
pd.read_csv(archivo, na_values=["N/A", "null", "-"], decimal=",", thousands=".")
```

## 7 · Validación y reporte de calidad

Una forma escalable: escribir cada regla como una **máscara booleana de filas válidas** y generar un reporte.

In [ ]:
UFS = {"AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA", "PB",
       "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"}


def reporte_calidad(df: pd.DataFrame, reglas: dict, id_col: str) -> pd.DataFrame:
  filas = []
  for nombre, valido in reglas.items():
    malos = df.loc[~valido.fillna(False), id_col]
    filas.append({
      "regla": nombre,
      "invalidos": len(malos),
      "pct": round(len(malos) / len(df) * 100, 2),
      "ejemplos": malos.head(3).tolist(),
    })
  return pd.DataFrame(filas).set_index("regla")


reglas_sellers = {
  "id_hex32": sellers["seller_id"].str.fullmatch(r"[0-9a-f]{32}"),
  "zip_5_digitos": sellers["seller_zip_code_prefix"].str.fullmatch(r"\d{5}"),
  "uf_valida": sellers["seller_state"].isin(UFS),
  "ciudad_solo_letras": sellers["seller_city"].str.fullmatch(r"[a-z]+(?:[ '\-][a-z]+)*"),
  "id_unico": ~sellers["seller_id"].duplicated(keep=False),
}
reporte_calidad(sellers, reglas_sellers, "seller_id")

In [ ]:
sellers.loc[~reglas_sellers["ciudad_solo_letras"], ["seller_id", "seller_city", "seller_state"]]

### Duplicados "ocultos"

`duplicated` sólo detecta copias **exactas**. Después de normalizar aparecen duplicados que antes no se veían:

In [ ]:
print("Títulos distintos (tal cual)        :", titulos.nunique())
print("Títulos distintos (llave canónica)  :", llave_canonica(titulos.dropna()).str.replace(r"[^\w\s]", "", regex=True).str.strip().nunique())

## 8 · Rendimiento con texto

El dtype `str` (respaldado por PyArrow) hace los métodos `.str` muy eficientes. Compara con `apply` sobre datos repetidos:

In [ ]:
grande = pd.concat([customers["customer_city"]] * 2000, ignore_index=True)
print(f"{len(grande):,} filas")

In [ ]:
%timeit grande.str.upper()

In [ ]:
%timeit grande.apply(str.upper)

Y cuando hay **pocos valores distintos repetidos muchas veces**, el truco del Módulo 01: limpiar las **categorías** (unas cientos) en lugar de las filas (cientos de miles).

In [ ]:
grande_cat = grande.astype("category")
%timeit grande_cat.cat.rename_categories(llave_canonica(grande_cat.cat.categories.to_series()).values)

In [ ]:
%timeit llave_canonica(grande)

> 💡 `rename_categories` exige que las categorías resultantes sean **únicas**. Si la limpieza junta dos variantes en una misma llave, usa `grande_cat.map(dict_de_limpieza)` (también opera sobre las categorías) o `astype(str)` primero.

---
# ✍️ Ejercicios

Variables y funciones disponibles: `reviews`, `customers`, `sellers`, `geo`, `traduccion`, `quitar_acentos`, `llave_canonica`, `UFS`, `reporte_calidad`.

### Ejercicio 1 · Normalizar títulos ⭐

1. `titulo_norm`: Series con `review_comment_title` normalizado: minúsculas, sin acentos, sin puntuación, espacios colapsados y sin espacios extremos. Los faltantes siguen siendo faltantes. Los títulos que queden vacíos (`""`) también deben ser faltantes.
2. `n_antes`, `n_despues`: valores distintos antes y después.
3. `top_titulo`: el título normalizado más frecuente.
4. `variantes_otimo`: lista **ordenada** de los títulos **originales** distintos cuyo normalizado es `"otimo"` u `"otima"`.

In [ ]:
# TU CÓDIGO AQUÍ
titulo_norm = ...
n_antes, n_despues = ..., ...
top_titulo = ...
variantes_otimo = ...

In [ ]:
# Comprobación
if pendiente(titulo_norm, n_antes, n_despues, top_titulo, variantes_otimo):
  print("⏳ Ejercicio 1 pendiente: resuélvelo en la celda de arriba")
else:
  assert len(titulo_norm) == len(reviews)
  assert titulo_norm.isna().sum() >= reviews["review_comment_title"].isna().sum()
  assert titulo_norm.dropna().str.fullmatch(r"[a-z0-9_]+( [a-z0-9_]+)*").all(), "Revisa acentos, puntuación o espacios"
  assert (n_antes, n_despues) == (30, 21), (n_antes, n_despues)
  assert top_titulo == "recomendo", top_titulo
  assert variantes_otimo == ["Otimo", "Otimo. ", "Ótima ", "Ótimo", "Ótimo "], variantes_otimo
  print("✅ Ejercicio 1 correcto")

### Ejercicio 2 · Etiquetado por regex ⭐⭐

Sobre los **mensajes** (`review_comment_message`) normalizados (minúsculas, sin acentos):

1. `tags`: DataFrame booleano (mismo índice que `reviews`) con estas columnas:
   - `prazo`: menciona "prazo" (como palabra completa).
   - `nao_recebi`: contiene "nao recebi" o "nao chegou".
   - `exclama`: contiene **dos o más** signos `!` seguidos.
   - `menciona_numero`: contiene al menos un número (secuencia de dígitos).
   Los mensajes faltantes cuentan como `False` en todas.
2. `score_medio`: Series indexada por el nombre de cada etiqueta con el **promedio** de `review_score` de las reseñas que la tienen.
3. `peor_tag`: la etiqueta con menor `score_medio`.

In [ ]:
# TU CÓDIGO AQUÍ
tags = ...
score_medio = ...
peor_tag = ...

In [ ]:
# Comprobación
if pendiente(tags, score_medio, peor_tag):
  print("⏳ Ejercicio 2 pendiente: resuélvelo en la celda de arriba")
else:
  assert list(tags.columns) == ["prazo", "nao_recebi", "exclama", "menciona_numero"]
  assert tags.index.equals(reviews.index) and (tags.dtypes == bool).all()
  assert tags.sum().to_dict() == {"prazo": 27, "nao_recebi": 11, "exclama": 5, "menciona_numero": 13}, tags.sum().to_dict()
  assert peor_tag == "nao_recebi", peor_tag
  assert np.isclose(score_medio["prazo"], 4.3333, atol=0.01), score_medio["prazo"]
  print("✅ Ejercicio 2 correcto")

### Ejercicio 3 · Ciudades de clientes y vendedores ⭐⭐

1. `canon_cli` y `canon_vend`: conjuntos (`set`) de tuplas `(ciudad_canonica, uf)` presentes en `customers` y en `sellers`.
2. `n_ciudades_ambos`: número de pares `(ciudad, uf)` con al menos un cliente **y** un vendedor.
3. `n_vend_sin_cli`: número de **vendedores** (filas de `sellers`) cuya ciudad no tiene ningún cliente en la muestra (comparando por llave canónica **y** estado).
4. `variantes_sp`: lista ordenada de las escrituras **originales** distintas que, juntando las columnas de ciudad de `customers`, `sellers` y `geo`, se canonicalizan como `"sao paulo"`.

In [ ]:
# TU CÓDIGO AQUÍ
canon_cli, canon_vend = ..., ...
n_ciudades_ambos = ...
n_vend_sin_cli = ...
variantes_sp = ...

In [ ]:
# Comprobación
if pendiente(canon_cli, canon_vend, n_ciudades_ambos, n_vend_sin_cli, variantes_sp):
  print("⏳ Ejercicio 3 pendiente: resuélvelo en la celda de arriba")
else:
  assert isinstance(canon_cli, set) and isinstance(canon_vend, set)
  assert ("sao paulo", "SP") in canon_cli and ("sao paulo", "SP") in canon_vend
  assert n_ciudades_ambos == 35, n_ciudades_ambos
  assert n_vend_sin_cli == 90, n_vend_sin_cli
  assert variantes_sp == ["sao paulo", "são paulo"], variantes_sp
  print("✅ Ejercicio 3 correcto")

### Ejercicio 4 · Precios en formato brasileño ⭐⭐

Una exportación (sintética) de precios de vendedores:

In [ ]:
export = pd.Series(
  ["R$ 1.234,56", "R$89,90", "  129,00", "R$ 2.500", "grátis", "GRATIS", "consultar",
   "R$ 0,99", "N/A", "", None, "R$ 12.345.678,90", "15.5", "R$ -3,00"],
  name="preco",
)

1. `precio`: Series `float` con el valor numérico, siguiendo estas reglas:
   - `"grátis"` (con o sin acento, cualquier mayúscula) → `0.0`.
   - Formato brasileño: punto = miles, coma = decimales. Un número **sin coma** se interpreta como entero con separador de miles (`"R$ 2.500"` → `2500.0`, `"15.5"` → `155.0`).
   - Lo que no se pueda interpretar (incluidos casi-nulos) → `NaN`.
2. `invalidos`: lista con los **textos originales** (sin faltantes reales) que quedaron en `NaN`.
3. `negativos`: número de precios negativos (para revisar).

In [ ]:
# TU CÓDIGO AQUÍ
precio = ...
invalidos = ...
negativos = ...

In [ ]:
# Comprobación
if pendiente(precio, invalidos, negativos):
  print("⏳ Ejercicio 4 pendiente: resuélvelo en la celda de arriba")
else:
  assert precio.dtype == "float64" and len(precio) == len(export)
  esperado = [1234.56, 89.90, 129.00, 2500.0, 0.0, 0.0, np.nan, 0.99, np.nan, np.nan, np.nan, 12345678.90, 155.0, -3.0]
  assert np.allclose(precio.values, esperado, equal_nan=True), precio.tolist()
  assert invalidos == ["consultar", "N/A", ""], invalidos
  assert negativos == 1
  print("✅ Ejercicio 4 correcto")

### Ejercicio 5 · Reto: reporte de calidad de `customers` ⭐⭐⭐

Escribe `reglas_cli` (diccionario nombre → máscara de **filas válidas**) y genera `rep_cli = reporte_calidad(customers, reglas_cli, "customer_id")` con estas reglas, **en este orden**:

| regla | válido si… |
|---|---|
| `id_hex32` | `customer_id` son 32 caracteres hexadecimales en minúscula |
| `unique_id_hex32` | lo mismo para `customer_unique_id` |
| `zip_5_digitos` | el prefijo postal son exactamente 5 dígitos |
| `uf_valida` | `customer_state` está en `UFS` |
| `ciudad_canonica` | `customer_city` ya es igual a su `llave_canonica` (no requiere limpieza) |
| `zip_coherente_uf` | el **primer dígito** del prefijo corresponde a la región del estado según la tabla de abajo |
| `ciudad_en_geo` | la llave canónica de la ciudad existe entre las llaves canónicas de `geo` **del mismo estado** |

Primer dígito del CEP por estado (simplificado):

In [ ]:
DIGITO_UF = {
  "0": {"SP"}, "1": {"SP"}, "2": {"RJ", "ES"}, "3": {"MG"}, "4": {"BA", "SE"},
  "5": {"PE", "AL", "PB", "RN"}, "6": {"CE", "PI", "MA", "PA", "AP", "AM", "RR", "AC"},
  "7": {"DF", "GO", "TO", "MT", "MS", "RO"}, "8": {"PR", "SC"}, "9": {"RS"},
}

# TU CÓDIGO AQUÍ
reglas_cli = ...
rep_cli = ...

In [ ]:
# Comprobación
if pendiente(reglas_cli, rep_cli):
  print("⏳ Ejercicio 5 pendiente: resuélvelo en la celda de arriba")
else:
  assert list(rep_cli.index) == ["id_hex32", "unique_id_hex32", "zip_5_digitos", "uf_valida",
                                 "ciudad_canonica", "zip_coherente_uf", "ciudad_en_geo"]
  assert rep_cli["invalidos"].to_dict() == {"id_hex32": 0, "unique_id_hex32": 0, "zip_5_digitos": 0, "uf_valida": 0, "ciudad_canonica": 4, "zip_coherente_uf": 0, "ciudad_en_geo": 16}, rep_cli["invalidos"].to_dict()
  print("✅ Ejercicio 5 correcto")

---
# ✅ Soluciones

### Solución 1

In [ ]:
titulo_norm = (
  quitar_acentos(reviews["review_comment_title"].str.lower())
  .str.replace(r"[^\w\s]", " ", regex=True)
  .str.replace(r"\s+", " ", regex=True)
  .str.strip()
  .replace("", np.nan)
)
n_antes = reviews["review_comment_title"].nunique()
n_despues = titulo_norm.nunique()
top_titulo = titulo_norm.value_counts().idxmax()
variantes_otimo = sorted(reviews.loc[titulo_norm.isin(["otimo", "otima"]), "review_comment_title"].unique())
(n_antes, n_despues), top_titulo, variantes_otimo

*Nota:* reemplazamos la puntuación por **espacio** (no por vacío) para no pegar palabras ("bom!!muito" → "bom muito"), y luego colapsamos espacios. El `.replace("", np.nan)` final evita contar el texto vacío como un título válido.

### Solución 2

In [ ]:
m = quitar_acentos(reviews["review_comment_message"].str.lower())

tags = pd.DataFrame({
  "prazo": m.str.contains(r"\bprazo\b", na=False),
  "nao_recebi": m.str.contains(r"nao (?:recebi|chegou)", na=False),
  "exclama": m.str.contains(r"!{2,}", na=False),
  "menciona_numero": m.str.contains(r"\d+", na=False),
})

score_medio = pd.Series({c: reviews.loc[tags[c], "review_score"].mean() for c in tags.columns})
peor_tag = score_medio.idxmin()
tags.sum(), score_medio.round(2), peor_tag

*Nota:* `\b` evita que "prazo" coincida dentro de "prazos" o "aprazo"... salvo que quieras eso. Una alternativa vectorizada para `score_medio`: `tags.mul(reviews["review_score"], axis=0).sum() / tags.sum()`.

### Solución 3

In [ ]:
canon_cli = set(zip(llave_canonica(customers["customer_city"]), customers["customer_state"]))
canon_vend = set(zip(llave_canonica(sellers["seller_city"]), sellers["seller_state"]))

n_ciudades_ambos = len(canon_cli & canon_vend)

pares_vend = pd.Series(list(zip(llave_canonica(sellers["seller_city"]), sellers["seller_state"])),
                       index=sellers.index)
n_vend_sin_cli = int((~pares_vend.isin(canon_cli)).sum())

todas = pd.concat([customers["customer_city"], sellers["seller_city"], geo["geolocation_city"]],
                  ignore_index=True)
variantes_sp = sorted(todas[llave_canonica(todas) == "sao paulo"].unique())
n_ciudades_ambos, n_vend_sin_cli, variantes_sp

*Notas:* comparar por `(ciudad, uf)` evita confundir ciudades homónimas de estados distintos (en Brasil hay varias, como "Bom Jesus"). Para el paso 3, `isin` acepta un `set` de tuplas cuando la Series contiene tuplas. La operación de conjuntos `&` es una forma muy rápida de intersectar llaves únicas.

### Solución 4

In [ ]:
texto = export.str.strip()
es_gratis = quitar_acentos(texto.str.lower()).eq("gratis")

numerico = pd.to_numeric(
  texto.str.replace(r"R\$|\s", "", regex=True)
       .str.replace(".", "", regex=False)
       .str.replace(",", ".", regex=False),
  errors="coerce",
)
precio = numerico.mask(es_gratis.fillna(False), 0.0).astype("float64")

invalidos = export[precio.isna() & export.notna()].tolist()
negativos = int((precio < 0).sum())
pd.DataFrame({"original": export, "precio": precio}), invalidos, negativos

*Notas:* `mask(cond, valor)` reemplaza donde la condición es `True` (lo opuesto de `where`). `pd.to_numeric` sobre un texto vacío da `NaN` y sobre `"-3.00"` da `-3.0`; un reporte de negativos es una **validación de negocio**, no de formato. La regla "sin coma = entero con miles" convierte `"15.5"` en `155`: documentar esas decisiones es tan importante como el código.

### Solución 5

In [ ]:
uf = customers["customer_state"]
zip_ = customers["customer_zip_code_prefix"]
canon = llave_canonica(customers["customer_city"])

geo_canon_uf = set(zip(geo["geolocation_state"], llave_canonica(geo["geolocation_city"])))

reglas_cli = {
  "id_hex32": customers["customer_id"].str.fullmatch(r"[0-9a-f]{32}"),
  "unique_id_hex32": customers["customer_unique_id"].str.fullmatch(r"[0-9a-f]{32}"),
  "zip_5_digitos": zip_.str.fullmatch(r"\d{5}"),
  "uf_valida": uf.isin(UFS),
  "ciudad_canonica": customers["customer_city"].eq(canon),
  "zip_coherente_uf": pd.Series([u in DIGITO_UF.get(z[:1], set()) for z, u in zip(zip_, uf)],
                                index=customers.index),
  "ciudad_en_geo": pd.Series([(u, c) in geo_canon_uf for u, c in zip(uf, canon)], index=customers.index),
}
rep_cli = reporte_calidad(customers, reglas_cli, "customer_id")
rep_cli

*Notas:*

- Algunas reglas no tienen un método vectorizado directo; una **comprensión de listas sobre `zip`** es legible y suficientemente rápida para cientos de miles de filas.
- `ciudad_canonica` marca como "inválidas" las ciudades con guion o apóstrofo (`mogi-guacu`, `herval d'oeste`): no son errores, sino datos que **necesitan** limpieza antes de unir. Un reporte de calidad mezcla errores y advertencias; vale la pena distinguirlos en el nombre de la regla.
- `ciudad_en_geo` depende de la cobertura de la muestra de geolocalización: con el dataset completo, casi todas pasarían.

---
## Resumen del módulo

- Los métodos `.str` son vectorizados y propagan faltantes; usa `na=False` en filtros.
- Limpieza por pasos: `strip` → `lower` → quitar acentos (`normalize("NFKD")`) → puntuación → espacios.
- Regex: `fullmatch` para validar, `extract`/`extractall` para extraer, `replace` con grupos o funciones.
- **Llave canónica** en ambos lados antes de comparar o unir texto; `difflib` sólo para **sugerir**.
- Casi-nulos → `NaN` explícito; números locales con `replace` + `pd.to_numeric(errors="coerce")`, o `decimal=`/`thousands=` al leer.
- Validación como **máscaras de filas válidas** + un reporte reutilizable.
- Con muchos valores repetidos, limpia las **categorías**, no las filas.

**Siguiente:** Módulo 08 · Method chaining y pipelines: `assign`, `pipe`, `query` y `eval`.